# RLM single-GPU training smoke test
Select a GPU runtime first. This notebook installs the checked-out project, its pinned Colab extra, and the optional Hugging Face dataset adapter; all training logic lives in `rlm_train.colab`.

In [ ]:
%pip install -e . -e './training[colab,hub-datasets]'

## Optional: create deterministic AIME24 and MATH-500 splits
The source revisions and split salts are pinned in `rlm_train.benchmarks`. Artifacts are written to Drive, byte-validated on reruns, and exposed below as ordinary notebook variables. The defaults are project-local AIME24 24/6 and MATH-500 400/100 partitions, not official upstream train/test splits.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DATASET_ROOT = Path('/content/drive/MyDrive/rlm-ib-datasets')

In [ ]:
from rlm_train.benchmarks import prepare_aime24_splits

AIME24_SPLITS = prepare_aime24_splits(DATASET_ROOT / 'aime24')
AIME24_VARIABLES = AIME24_SPLITS.notebook_variables('AIME24')
globals().update(AIME24_VARIABLES)
AIME24_VARIABLES

In [ ]:
from rlm_train.benchmarks import prepare_math500_splits

MATH500_SPLITS = prepare_math500_splits(DATASET_ROOT / 'math500')
MATH500_VARIABLES = MATH500_SPLITS.notebook_variables('MATH500')
globals().update(MATH500_VARIABLES)
MATH500_VARIABLES

In [ ]:
# Optional for an API-judge run; the secret value is never written to config or output.
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
!rlm-train-colab training/configs/colab-smoke.toml

## Build separate pure-SDPO AIME24 and MATH-500 runs
Each config uses its dataset's train partition for SDPO updates and its disjoint test partition for ordinary final-answer evaluation. `policy_weight=0` and `sdpo_weight=1`, so GRPO contributes no loss. The default fake judge validates the local pipeline but gives generic feedback; select the OpenAI judge and set its secret above for target-aware feedback.

In [ ]:
from rlm_train.colab import (
    build_benchmark_sdpo_config,
    write_colab_run_config,
)
from rlm_train.colab.config import JudgeConfig

SDPO_JUDGE = JudgeConfig()  # deterministic fake: pipeline validation only
# For target-aware feedback, replace the line above with an explicit API model:
# SDPO_JUDGE = JudgeConfig(provider='openai', model='YOUR_MODEL', model_revision='YOUR_REVISION')

AIME24_CONFIG = build_benchmark_sdpo_config(
    AIME24_SPLITS, run_name='aime24-sdpo', max_optimizer_steps=100, judge=SDPO_JUDGE
)
MATH500_CONFIG = build_benchmark_sdpo_config(
    MATH500_SPLITS, run_name='math500-sdpo', max_optimizer_steps=100, judge=SDPO_JUDGE
)
AIME24_CONFIG_PATH = write_colab_run_config(AIME24_CONFIG, '/content/aime24-sdpo.json')
MATH500_CONFIG_PATH = write_colab_run_config(MATH500_CONFIG, '/content/math500-sdpo.json')
AIME24_CONFIG_PATH, MATH500_CONFIG_PATH

### AIME24 SDPO run
Run this cell by itself. Every completed optimizer step prints a JSON metrics record; a healthy pure-SDPO step has `loss/sdpo > 0`, `loss/policy = 0`, `tokens/active_policy = 0`, and `optimizer/gradient_norm > 0`.

In [ ]:
%cd /content/rlm-ib
!rlm-train-colab {AIME24_CONFIG_PATH}

### MATH-500 SDPO run
Run this separately from AIME24. It writes to a different Drive run directory.

In [ ]:
%cd /content/rlm-ib
!rlm-train-colab {MATH500_CONFIG_PATH}

The first command remains the synthetic GRPO smoke test; preparing datasets does not silently change it. The AIME24 and MATH-500 configs are independent pure-SDPO runs. To resume either latest checkpoint, rerun its command with `--resume`; an explicit checkpoint directory may follow the flag.